In [3]:
## Import necessary libraries
import numpy as np
import pandas as pd
import requests
import os
import sys
import json

## Impoer  --- IGNORE ---


In [58]:
file_dir = 'd:\\personal\\learn\\practice\\git\\data-engineer\\data_sources\\'

df = pd.read_csv(file_dir + 'marketing_campaign_data_messy.csv')

## size of dataframe
print(f'Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns')    

Loaded Dataset: 2020 rows, 12 columns


In [23]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


In [17]:
## ============================================
## STEP 1: HEADER CLEANING
## ============================================

In [59]:
## column names to lowercase
print('before cleaning: ', df.columns)
df.columns = df.columns.str.strip().str.lower()
print('after cleaning :', df.columns)

before cleaning:  Index([' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel',
       'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks',
       'Campaign_Tag'],
      dtype='object')
after cleaning : Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks',
       'campaign_tag'],
      dtype='object')


In [19]:
## ============================================
## STEP 2: TYPE CONVERSION & CURRENCY CLEANING
## ============================================

In [60]:
dirty_spend_mask = df['spend'].astype(str).str.contains(r'[\$,]')
print(f'Number of dirty spend entries: {dirty_spend_mask.sum()}')
print(df.loc[dirty_spend_mask, ['campaign_id','spend']].head(3))

df['spend'] = df['spend'].replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')   

## ============================================
print(df.loc[dirty_spend_mask, ['campaign_id','spend']].head(3))

Number of dirty spend entries: 298
   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [20]:
## ============================================
## STEP 3: CETEGORICAL TYPOS (FUZZY LOGIC)
## ============================================

In [67]:
## Get unique values in 'channel' column
df.channel.unique()

array(['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', 'E-mail',
       nan, 'Gogle', 'Tik_Tok', 'Facebok', 'Insta_gram'], dtype=object)

In [68]:
## Create a mapping dictionary to correct typos
channel_map = {
    'Email': 'Email', ## correct value
    'E-mail': 'Email', ## typo
    'TikTok': 'TikTok', ## correct value
    'Tik_Tok': 'TikTok',    ## typo 
    'Facebook': 'Facebook', ## correct value
    'Facebok': 'Facebook',  ## typo
    'Google Ads': 'Google Ads', ## correct value
    'Gogle': 'Google Ads',  ## typo
    'Instagram': 'Instagram',   ## correct value
    'Insta_gram': 'Instagram',  ## typo
    'N/A': np.nan ## convert N/A to NaN 
}

In [69]:
## Apply the mapping to the 'channel' column
df['channel'] = df['channel'].map(channel_map)
## Verify the changes
df.channel.unique()

array(['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan],
      dtype=object)

In [21]:
## ============================================
## STEP 4: HANDLING MIXED BOOLEAN & NUMERIC VALUES
## ============================================

In [71]:
## Get unique values in 'active' column
df.active.unique()

array(['Y', '0', 'No', 'True', 'Yes', '1', 'False'], dtype=object)

In [73]:
active_map = {
    'Yes': True, ## correct value
    'No': False, ## correct value
    'Y': True,  ## typo
    'N': False, ## typo
    '1': True,  ## numeric representation
    '0': False, ## numeric representation
    1: True,    ## numeric representation
    0: False,   ## numeric representation
    True: True, ## correct value
    False: False,   ## correct value
    'N/A': np.nan   ## convert N/A to NaN
}

In [74]:
## Apply the mapping to the 'active' column
df.active = df['active'].map(active_map)

## Verify the changes
df.active.unique()

array([True, False, nan], dtype=object)

In [22]:
## ============================================
## STEP 5: DATE PARSING AND STANDARDIZATION
## ============================================

In [ ]:
## 
print(df['start_date'].dtype)

object


In [ ]:
## 
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce', format='%Y-%m-%d')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce', format='%Y-%m-%d')

In [85]:
df['start_date'].unique()

<DatetimeArray>
[                'NaT', '2023-10-30 00:00:00', '2023-05-23 00:00:00',
 '2023-04-09 00:00:00', '2023-12-15 00:00:00', '2023-10-18 00:00:00',
 '2023-04-28 00:00:00', '2023-01-17 00:00:00', '2023-10-24 00:00:00',
 '2023-02-11 00:00:00',
 ...
 '2023-01-28 00:00:00', '2023-12-24 00:00:00', '2023-02-13 00:00:00',
 '2023-12-02 00:00:00', '2023-03-09 00:00:00', '2023-06-02 00:00:00',
 '2023-10-19 00:00:00', '2023-03-18 00:00:00', '2023-02-27 00:00:00',
 '2023-08-27 00:00:00']
Length: 124, dtype: datetime64[ns]

In [ ]:
## 
print(df['start_date'].dtype)

datetime64[ns]


In [ ]:
## ============================================
## STEP 6: LOGICAl INTEGRITY CHECKS (CLICKS vs IMPRESSIONS)
## ============================================

In [87]:
## Drop duplicate columns if any
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
## Identify rows where clicks exceed impressions
clicks_exceed_mask = df['clicks'] > df['impressions']

print(f'Number of rows where clicks exceed impressions: {clicks_exceed_mask.sum()}')


Number of rows where clicks exceed impressions: 0


In [ ]:
## ============================================
## STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)
## ============================================

In [91]:
time_travel_mask = df['end_date'] < df['start_date']
print(f'Number of rows with time travel issues: {time_travel_mask.sum()}')
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

Number of rows with time travel issues: 39
    campaign_id start_date   end_date
124   CMP-00125 2023-04-01 2023-02-02
195   CMP-00196 2023-08-04 2023-04-21
241   CMP-00242 2023-10-01 2023-01-11


In [93]:
df.loc[time_travel_mask,'end_date'] = df.loc[time_travel_mask, 'start_date']+ pd.Timedelta(days=30)

In [94]:
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

    campaign_id start_date   end_date
124   CMP-00125 2023-04-01 2023-05-01
195   CMP-00196 2023-08-04 2023-09-03
241   CMP-00242 2023-10-01 2023-10-31


In [95]:
## ============================================
## STEP 8: HANDLING OUTIERS (WINSORIZING)
## ============================================

In [97]:
q1 = df['spend'].quantile(0.25)
q3 = df['spend'].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 3 * iqr
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

Lower Bound: -2332.5799999999995, Upper Bound: 8603.537499999999


In [104]:
spend_outlier_mask = df['spend'] > upper_bound
print(f'Number of spend outliers: {spend_outlier_mask.sum()}')
print(df.loc[spend_outlier_mask, ['campaign_id', 'spend']].head(3))

Number of spend outliers: 6
     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00


In [107]:
## 
df.loc[spend_outlier_mask, 'spend'] = upper_bound

In [108]:
print(df.loc[spend_outlier_mask, ['campaign_id', 'spend']].head(3))

     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


In [ ]:
## ============================================
## STEP 9: STRING PARSING (FEATURE EXTRACTION)
## ============================================

In [110]:
print(df['campaign_name'].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object


In [117]:
## Campaign Quarter
df['campaign_qtr'] = df['campaign_name'].str.split('_', expand=True)[0]

## Campaign season
df['campaign_season'] = df['campaign_name'].str.split('_', expand=True)[1]


In [118]:
df.head(10)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,campaign_qtr,campaign_season
0,CMP-00001,Q4_Summer_CMP-00001,NaT,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Q4,Summer
1,CMP-00002,Q1_Launch_CMP-00002,NaT,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Q1,Launch
2,CMP-00003,Q3_Winter_CMP-00003,NaT,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Q3,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,NaN,TI,Q1,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,NaT,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Q2,Winter
5,CMP-00006,Q4_BlackFriday_CMP-00006,NaT,2023-10-28,Instagram,83386,2643,2697.03,NaN,True,IN,Q4,BlackFriday
6,CMP-00007,Q3_Launch_CMP-00007,NaT,2023-10-23,Facebook,38194,1135,1232.76,178.0,True,FA,Q3,Launch
7,CMP-00008,Q4_Launch_CMP-00008,2023-05-23,2023-05-28,Instagram,88498,1173,865.70,127.0,True,IN,Q4,Launch
8,CMP-00009,Q4_BlackFriday_CMP-00009,NaT,2023-04-01,Google Ads,45131,1179,1046.18,104.0,True,GO,Q4,BlackFriday
9,CMP-00010,Q2_Winter_CMP-00010,NaT,2023-04-01,Email,61263,1153,1623.56,NaN,False,EM,Q2,Winter


In [121]:
df['campaign_season'].value_counts()

campaign_season
Winter         518
Launch         512
BlackFriday    505
Summer         485
Name: count, dtype: int64